# Tutorial 1: Quick Start

This notebook mirrors the quick-start flow from the getting started guide.



## What is `floodsr`?

`floodsr` is a flood-depth resolution enhancement tool.
It takes a low-resolution depth raster and reconstructs a higher-resolution result using terrain context from a DEM.

# Using this tutorial

There are three common ways to run this tutorial (i.e., *execution contexts*):
- **command line (CLI)**: proceed by copy/pasting the below commands from your web-browser into your terminal. NOTE for most terminals you'll need to remove the prefix `!` from each command.
- **local notebook (Jupyter)**: Use the <i class="fas fa-download"></i> button to save this notebook as an `.ipynb` file to your local machine (may need to click *save as*) and open it with your local Jupyter kernel. see {ref}`_basic_install_local_notebook` for for details.
- **hosted notebook (Colab)**: Use the <i class="fas fa-rocket"></i> button to open this tutorial on Google Colab.


# Install `floodsr`

The appropriate installation steps depend on your execution context, which is described in detail in :ref:`_basic_install`.
Here are the basic steps for each:
- **command line (CLI)**: first, ensure pipx is installed and on PATH, then run `pipx install floodsr`. see {ref}`_basic_install_cli` for more details.
- **local notebook (Jupyter)**: same as above. then either follow the additional steps in {ref}`_basic_install_notebook` to properly set up your environment, or be lazy and uncomment the below Jupyter cell and run.
- **hosted notebook (Colab)**: uncomment the below colab cell and run. see {ref}`_basic_install_hosted_notebook` for details.

In [ ]:
# command line (CLI) install. run `pipx install floodsr`

Local notebook (Jupyter) magic install

In [ ]:

# %pip install -q floodsr

Hosted notebook (Colab) install

In [ ]:
# !python -m pip install -q floodsr

### prove the install

run the below command to confirm the install worked and `floodsr` is on your PATH:

In [3]:
!floodsr --help

usage: floodsr [-h] [-v] [-q] [--log-level {DEBUG,INFO,WARNING,ERROR}]
               {models,tohr,doctor} ...

Run FloodSR model, cache, and runtime utility commands.

positional arguments:
  {models,tohr,doctor}
    models              List manifest models or fetch cached model weights.
    tohr                Run one super-resolution pass for a low-res depth
                        raster.
    doctor              Report runtime dependency and provider diagnostics.

options:
  -h, --help            show this help message and exit
  -v, --verbose         Increase logging verbosity (repeatable).
  -q, --quiet           Decrease logging verbosity (repeatable).
  --log-level {DEBUG,INFO,WARNING,ERROR}
                        Explicit log level override.


## Download test data

Before running commands, download a small example dataset into your current working directory.

In [ ]:
from urllib.request import urlretrieve

urlretrieve(
    "https://github.com/cefect/floodsr/releases/download/v0.0.3/hires002_dem.tif",
    "hires002_dem.tif",
)
urlretrieve(
    "https://github.com/cefect/floodsr/releases/download/v0.0.3/lowres032.tif",
    "lowres032.tif",
)


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0


  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0


  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0



100  791k  100  791k    0     0  1775k      0 --:--:-- --:--:-- --:--:-- 1775k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0


  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0



100  2898  100  2898    0     0  22878      0 --:--:-- --:--:-- --:--:-- 22878


## List available models

Before running inference, inspect the manifest to see which model versions are available.

In [4]:
!floodsr models list

ResUNet_16x_DEM	model_infer.onnx	https://github.com/cefect/floodsr/releases/download/v2026.02.19/model_infer.onnx


## Fetch model weights

Fetch the default model into the local cache.

In [5]:
!floodsr models fetch --no-progress ResUNet_16x_DEM


INFO:floodsr.model_registry:attempting unauthenticated model download from
    https://github.com/cefect/floodsr/releases/download/v2026.02.19/model_infer.onnx


/root/.cache/floodsr/ResUNet_16x_DEM/model_infer.onnx


## Run `tohr` with fetched HRDEM data

If your low-resolution raster falls within HRDEM coverage, you can fetch the high-resolution DEM automatically.

In [6]:
!floodsr tohr --in lowres032.tif --fetch-hrdem

INFO:floodsr.cli:starting DEM fetch
  source_id=hrdem
  stac_url=https://datacube.services.geo.ca/api
  collection=hrdem-mosaic-1m
  asset_key=dtm
  gdal_available=False
  force_tiling=False
  fetch_window_size=256
  memory_limit_gib=16.00
  stac_query_limit=200
  use_project_extent_filter=True
  use_cache=True
  tqdm_disable=False
  project_extent_url=https://maps-cartes.services.geo.ca/server_serveur/rest/services/NRCan/coverage_HRDEM_en/MapServer/4
  depth_lr_fp=
    /workspace/docs/user/notebooks/lowres032.tif


INFO:rasterio.session:boto3 not available, falling back to a DummySession.


INFO:rasterio.session:boto3 not available, falling back to a DummySession.
INFO:floodsr.cli:found 1 HRDEM item(s) intersecting low-res tile bounds after exact intersection filter
INFO:rasterio.session:boto3 not available, falling back to a DummySession.
INFO:floodsr.cli:raw fetch request grid: width=1,024, height=1,024, pixels=1,048,576, non_windowed_peak_estimate=0.01 GiB
INFO:rasterio.session:boto3 not available, falling back to a DummySession.


INFO:floodsr.cli:wrote fetched HRDEM tile to
    /tmp/floodsr_hrdem_output_b62ab9a1548c327a83fa6dd6.tif


INFO:floodsr.cli:loaded ORT model 'model_infer.onnx' with providers=['CPUExecutionProvider'] and scale=16


INFO:floodsr.cli:starting tohr inference with model_version=ResUNet_16x_DEM
model
    /root/.cache/floodsr/ResUNet_16x_DEM/model_infer.onnx
platform_depth_lr
    /tmp/floodsr-platform-prep-iqjdkwr_/lowres032_platform_depth.tif
platform_dem_hr
    /tmp/floodsr-platform-prep-iqjdkwr_/floodsr_hrdem_output_b62ab9a1548c327a83fa6dd6_platform_dem.tif
output
    /workspace/docs/user/notebooks/lowres032_sr.tif
INFO:floodsr.cli:platform-preprocessed inputs
  depth_lr shape=(32, 32) res=(32.0, 32.0) m/pix
  dem_hr shape=(1024, 1024) res=(1.0, 1.0) m/pix


INFO:floodsr.cli:model preprocessing complete
  scale=16 (HR/LR ratio)
  crs_policy=strict
  aligned depth shape=(32, 32) resampled=True
  aligned dem shape=(512, 512) raw_dem_shape=(1024, 1024)
  max_depth=5.0 dem_pct_clip=95.0
INFO:floodsr.cli:prepared inputs summary:
  aligned depth_lr shape=(32, 32) res=(32.0, 32.0) m/pix
  aligned dem_hr shape=(512, 512) res=(2.0, 2.0) m/pix
  max_depth=5.0
  dem_pct_clip=95.0
INFO:floodsr.cli:window config
  method=feather
  overlap_lr=8
  overlap_hr=128
  tile_size_lr=32
  tile_size_hr=512
INFO:floodsr.cli:running feather tiling over 1x1 grid
  stride_hr=384
  overlap_hr=128

feather pass:   0%|                                   | 0/1 [00:00<?, ?window/s]


feather pass: 100%|███████████████████████████| 1/1 [00:00<00:00, 11.78window/s]
INFO:floodsr.cli:post-resampling model output from (512, 512) to (1024, 1024) on raw DEM grid with bilinear interpolation.


INFO:floodsr.cli:finished tohr inference in 0.198s; wrote 4,624,434 bytes to
    /workspace/docs/user/notebooks/lowres032_sr.tif
/workspace/docs/user/notebooks/lowres032_sr.tif


## Run `tohr` with a local DEM

If you already have a local DEM, point `floodsr` at it directly.

In [7]:
!floodsr tohr --in lowres032.tif --dem hires002_dem.tif

INFO:floodsr.cli:loaded ORT model 'model_infer.onnx' with providers=['CPUExecutionProvider'] and scale=16


INFO:floodsr.cli:starting tohr inference with model_version=ResUNet_16x_DEM
model
    /root/.cache/floodsr/ResUNet_16x_DEM/model_infer.onnx
platform_depth_lr
    /tmp/floodsr-platform-prep-mmln2oee/lowres032_platform_depth.tif
platform_dem_hr
    /tmp/floodsr-platform-prep-mmln2oee/hires002_dem_platform_dem.tif
output
    /workspace/docs/user/notebooks/lowres032_sr.tif
INFO:floodsr.cli:platform-preprocessed inputs
  depth_lr shape=(32, 32) res=(32.0, 32.0) m/pix
  dem_hr shape=(512, 512) res=(2.0, 2.0) m/pix
INFO:floodsr.cli:model preprocessing complete
  scale=16 (HR/LR ratio)
  crs_policy=strict
  aligned depth shape=(32, 32) resampled=False
  aligned dem shape=(512, 512) raw_dem_shape=(512, 512)
  max_depth=5.0 dem_pct_clip=95.0
INFO:floodsr.cli:prepared inputs summary:
  aligned depth_lr shape=(32, 32) res=(32.0, 32.0) m/pix
  aligned dem_hr shape=(512, 512) res=(2.0, 2.0) m/pix
  max_depth=5.0
  dem_pct_clip=95.0
INFO:floodsr.cli:window config
  method=feather
  overlap_lr=8
  ove


feather pass: 100%|███████████████████████████| 1/1 [00:00<00:00, 18.72window/s]
INFO:floodsr.cli:finished tohr inference in 0.082s; wrote 1,175,741 bytes to
    /workspace/docs/user/notebooks/lowres032_sr.tif
/workspace/docs/user/notebooks/lowres032_sr.tif


## Next steps

For more detail, continue with {doc}`../user_guide` or inspect the full {doc}`../cli_reference`.